# Unit 2 solutions: popularity, quality, and bias

These reference solutions build the popularity count path, score it honestly against Unit 1's random floor, turn to the weighted-rating quality lens, and then measure the popularity bias the offline metric quietly rewards. Every count is computed on the `train` split only; `val` is the sealed holdout we grade against and `test` stays sealed tighter still. All random paths are seeded (`seed=0`), so the notebook runs top-to-bottom deterministically.


In [ ]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
interactions = pd.read_csv(data_dir / "interactions.csv.gz")
interactions_path = data_dir / "interactions.csv.gz"
rows = interactions.to_dict("records")
len(catalog), len(interactions)

## Exercise 1 - Count popularity on the training split only

Popularity is a count of positive (`label == 1`) interactions per book, taken on the `train` split alone. We filter, group by `item_id`, count, and sort descending. The final `splits_used` check proves the filter never touched `val` or `test`: after keeping only `train` rows, the only split present is `train`.

In [ ]:
train = interactions[interactions["split"] == "train"]
train_positives = train[train["label"] == 1]                 # keep label == 1 rows of train
counts = train_positives.groupby("item_id").size().sort_values(ascending=False)
top_item_id = int(counts.index[0])                           # most-read book
top_count = int(counts.iloc[0])                              # its positive count
splits_used = set(train_positives["split"])                  # expect {"train"}

print(f"most-read book: item {top_item_id} with {top_count} positive train interactions")
print(f"splits present after filtering: {splits_used}")

assert splits_used == {"train"}, "the count must never touch val or test"
assert top_count == int(counts.max()), "the most-read book must hold the max count"

## Exercise 2 - Fit the PopularityRetrievalPath and retrieve without repeats

`bookrec.PopularityRetrievalPath` ships this exact count. Its `fit` consumes the row mappings built in setup (leakage safety lives inside the code, not in our discipline), and `.counts` must match the by-hand counts from Exercise 1. `retrieve` honours `context["seen"]`, so neither already-read id leaks into the result, and because popularity is reader-independent, two different queries return the same list.

In [ ]:
popularity = bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)

by_hand = {int(i): int(c) for i, c in zip(counts.index, counts.values)}
counts_match = popularity.counts == by_hand

seen = {catalog_ids[0], catalog_ids[1]}
top5 = popularity.retrieve(query="reader-A", context={"seen": seen}, k=5)
top5_item_ids = [c.item_id for c in top5]
no_seen_leaked = not (seen & set(top5_item_ids))

top5_other = popularity.retrieve(query="reader-B", context={"seen": seen}, k=5)
reader_independent = top5_item_ids == [c.item_id for c in top5_other]

print(f"counts_match={counts_match}  top5={top5_item_ids}")
print(f"no_seen_leaked={no_seen_leaked}  reader_independent={reader_independent}")

assert counts_match, "the shipped counts must equal the by-hand counts"
assert no_seen_leaked and reader_independent

## Exercise 3 - Register the path and beat the random floor

Cold readers (from `cold_partitions.json`) must never be scored. We register the fitted path in a fresh `PathRegistry` (a duplicate name/artifact would raise), then score both `popularity` and a seeded `RandomRetrievalPath` floor on the `val` scoreboard with the cold-reader exclusion.

"Beats the floor" means popularity's `hit_rate_at_k` on the sealed `val` split is far above the random baseline's: a random pick of 10 of ~2000 books almost never lands on a reader's held-out positive, while recommending the crowd's favourites lands often. Popularity can win this *before we know anything about a reader* precisely because it ignores the reader -- the same crowd-pleasing list is a defensible first guess for everyone, which is why it is the universal cold-start fallback.

In [ ]:
cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])

registry = bookrec.PathRegistry()
registry.register(popularity)                                # name/artifact collision would raise
random_floor = bookrec.RandomRetrievalPath(catalog_ids, seed=0)

pop_board = bookrec.run_validation_scoreboard(
    popularity, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers
)
random_board = bookrec.run_validation_scoreboard(
    random_floor, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers
)
readers_scored = pop_board.readers
ratio = pop_board.hit_rate_at_k / random_board.hit_rate_at_k
beats_floor_explanation = (
    "Popularity's val hit@10 sits far above the random floor because recommending the "
    "crowd's most-read books lands on a reader's held-out positives far more often than a "
    "uniform pick of 10 of ~2000 items; it wins before we know the reader because the same "
    "crowd-pleasing list is a reasonable default for everyone (the cold-start fallback)."
)

print(f"readers scored:    {readers_scored}")
print(f"random hit@10:     {random_board.hit_rate_at_k:.4f}")
print(f"popularity hit@10: {pop_board.hit_rate_at_k:.4f}")
print(f"ratio:             {ratio:.1f}x the floor")

assert pop_board.hit_rate_at_k > 5 * random_board.hit_rate_at_k, "popularity must beat the floor by a wide margin"
assert ratio > 5.0 and readers_scored == 500

## Exercise 4 - Apply the weighted rating and predict the re-ranking

The weighted (Bayesian-shrinkage) rating is $\text{score} = \frac{v\,R + m\,C}{v + m}$, where $R$ is an item's own positive rate, $v$ its exposure, $C$ the global positive rate, and $m$ the prior strength in pseudo-counts. At $m = 0$ the score recovers the raw rate $R$; at $m = 10$ thin-evidence rates are pulled toward $C$.

Predicted re-ranking (raw rate $\to$ shrunk at $m=10$):

- Raw rate ranks the two *perfect-but-thin* items first: item 0 ($3/3 = 1.0$) and item 3 ($1/1 = 1.0$) tie at the top, then item 2 ($9000/10000 = 0.9$), then item 1 ($40/50 = 0.8$).
- At $m = 10$ the order flips to item 2, then item 1, then item 0, then item 3. The deeply-evidenced $0.9$ item (item 2, $v=10000$) barely moves and rises to the top, while the thin perfect items collapse toward $C=0.5$.
- The exercise asks about the **3-of-3 item (index 0)**: on raw rate it is tied for first (a perfect $1.0$ on only three observations), and under $m = 10$ shrinkage it collapses toward $C = 0.5$ and drops from **1st to 3rd**, overtaken by the deep items 2 and 1. That drop is the demotion we report (`demoted_thin_item = 0`). Item 3 ($1/1$) is even thinner and sinks all the way to last, but item 0 is our designated thin-but-perfect headline.

In [ ]:
positives = np.array([3, 40, 9000, 1])
exposures = np.array([3, 50, 10000, 1])
C = 0.5  # given global positive rate for this exercise

raw_scores = bookrec.weighted_rating(positives, exposures, m=0.0, global_rate=C)     # recovers R
shrunk_scores = bookrec.weighted_rating(positives, exposures, m=10.0, global_rate=C)
raw_order = [int(i) for i in np.argsort(-raw_scores, kind="stable")]
shrunk_order = [int(i) for i in np.argsort(-shrunk_scores, kind="stable")]
demoted_thin_item = 0          # the 3/3 book: tied-first on raw rate, sinks to 3rd under shrinkage
ranking_change = (
    "Raw rate puts the two perfect-but-thin items (0 and 3) first, then item 2 (0.9), then "
    "item 1 (0.8). At m=10 the deeply-evidenced item 2 rises to the top and both thin perfect "
    "items collapse toward C=0.5, so item 0 drops from 1st to 3rd and item 1 (deep enough) "
    "overtakes it."
)

print(f"raw rate R:     {np.round(raw_scores, 4)}  order {raw_order}")
print(f"weighted m=10:  {np.round(shrunk_scores, 4)}  order {shrunk_order}")

# the thin-vs-deep reversal: raw rate prefers 3/3 over 9000/10000, shrinkage reverses it
assert raw_scores[0] > raw_scores[2], "raw rate favours the thin 3/3 item"
assert shrunk_scores[0] < shrunk_scores[2], "shrinkage demotes the thin item below the deep one"
assert raw_order[0] == demoted_thin_item and shrunk_order.index(demoted_thin_item) > raw_order.index(demoted_thin_item)

## Exercise 5 - Score the quality ranking and explain the biased metric

We score every catalog item with the weighted rating on train exposures/positives ($m=10$, global rate $C$), wrap it in a reader-independent path that ranks by that score (the count path's structural twin), and put it on the same `val` scoreboard with the same cold-reader exclusion. It lands **far below popularity** -- about $0.016$ (hit@10), only ~1.3x the random floor (~$0.012$) and roughly 7x below the count path's ~$0.11$.

Why: ranking by positive rate floats **lightly-exposed** items (good rate, almost no evidence) to the top, and those are exactly the items the `val` holdout rarely contains. The holdout is built from interactions readers actually had, and exposure is **popularity-weighted** (taste-aware, but still dominated by popularity) -- readers mostly meet popular books, so the held-out positives are overwhelmingly already-popular items. A quality-first ranker therefore spends its slots on books that are anti-correlated with the holdout, so hit@k scores it far below popularity. That is popularity bias living inside the metric itself: the quality ranker is not broken, the measuring stick is bent.

In [ ]:
train = interactions[interactions["split"] == "train"]
train_positives = train[train["label"] == 1]
exposure_by = train.groupby("item_id").size()
positive_by = train_positives.groupby("item_id").size()
exposure_map = {int(i): int(v) for i, v in zip(exposure_by.index, exposure_by.values)}
positive_map = {int(i): int(v) for i, v in zip(positive_by.index, positive_by.values)}
item_exposures = np.array([exposure_map.get(i, 0) for i in catalog_ids])
item_positives = np.array([positive_map.get(i, 0) for i in catalog_ids])
C_global = float(item_positives.sum() / item_exposures.sum())

quality_scores = bookrec.weighted_rating(item_positives, item_exposures, m=10.0, global_rate=C_global)

class QualityRetrievalPath:
    name = "quality"

    def __init__(self, item_ids, scores):
        self._pairs = [(int(i), float(s)) for i, s in zip(item_ids, scores)]

    def retrieve(self, query, context, k):
        seen = set(context["seen"]) if "seen" in context else set()
        candidates = [
            bookrec.Candidate(item_id, score, "quality")
            for item_id, score in self._pairs
            if item_id not in seen
        ]
        return bookrec.rank(candidates, n=k)

quality_board = bookrec.run_validation_scoreboard(
    QualityRetrievalPath(catalog_ids, quality_scores),
    interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers,
)
quality_hit = quality_board.hit_rate_at_k
why_near_zero = (
    "Ranking by positive rate floats lightly-exposed items to the top, but the val holdout -- "
    "built from interactions readers actually had, and exposure is popularity-weighted -- is "
    "overwhelmingly already-popular items. So a quality-first ranker spends its slots on books "
    "anti-correlated with the holdout and hit@k scores it far below popularity. That is "
    "popularity bias living inside the metric: the ranker is fine, the measuring stick is bent."
)

print(f"global rate C = {C_global:.4f}")
print(f"popularity (count) hit@10: {pop_board.hit_rate_at_k:.4f}")
print(f"random floor hit@10:       {random_board.hit_rate_at_k:.4f}")
print(f"quality (weighted) hit@10: {quality_hit:.4f}")

assert quality_hit < 0.5 * pop_board.hit_rate_at_k, "the quality ranker must lose badly to the count path offline"

## Exercise 6 - Measure the bias: catalog coverage and head-share

We build the head set (top 10% of items by train positive count), reproduce the scoreboard's scored-reader set (non-cold readers with at least one unseen positive `val` item), and collect each one's top-k list for `popularity` and for the `random_floor`. `catalog_coverage` is the fraction of unique catalog items ever recommended; `head_share` is the fraction of recommendation slots landing in the head.

The random floor we just beat on accuracy is vastly *fairer* to the catalog: popularity serves almost none of it (coverage near $0.008$, about 16 of 2000 books) and fills every slot with head items (head-share $1.0$), while random spreads across ~92% of the catalog with a head-share near $0.10$. The gap is popularity bias: the count winner concentrates all exposure on a crowded few and starves the long tail.

In [ ]:
head = bookrec.head_ids_from_counts(popularity.counts, catalog_ids, fraction=0.10)

def scored_recommendations(path):
    """Reproduce the scoreboard's scored-reader set and collect each one's top-k list."""
    catalog_set = set(catalog_ids)
    seen_by = defaultdict(set)
    val_by = defaultdict(set)
    for row in rows:
        if row["item_id"] not in catalog_set or row["label"] != 1:
            continue
        if row["split"] == "train":
            seen_by[row["reader_id"]].add(row["item_id"])
        elif row["split"] == "val":
            val_by[row["reader_id"]].add(row["item_id"])
    recommendations = []
    for reader_id in sorted(val_by):
        seen = seen_by[reader_id]
        if reader_id in cold_readers or not (val_by[reader_id] - seen):
            continue
        recs = path.retrieve(reader_id, {"seen": seen}, 10)
        recommendations.append(recs)
    return recommendations

pop_recs = scored_recommendations(popularity)
random_recs = scored_recommendations(random_floor)
pop_coverage = bookrec.catalog_coverage(pop_recs, catalog_ids)
pop_head_share = bookrec.head_share(pop_recs, head)
random_coverage = bookrec.catalog_coverage(random_recs, catalog_ids)
random_head_share = bookrec.head_share(random_recs, head)
fairer_path_explanation = (
    "The random floor is far fairer to the catalog: popularity serves almost none of it and "
    "fills every slot with head items, while random spreads across most of the catalog. The "
    "gap exposes popularity bias -- the accuracy winner starves the long tail."
)

print(f"head set size: {len(head)}")
print(f"{'path':>12} {'coverage':>10} {'head-share':>11}")
print(f"{'popularity':>12} {pop_coverage:>10.4f} {pop_head_share:>11.4f}")
print(f"{'random':>12} {random_coverage:>10.4f} {random_head_share:>11.4f}")

assert pop_head_share == 1.0, "every popularity slot is a head item"
assert pop_coverage < random_coverage, "popularity is far less fair to the catalog than random"

## Exercise 7 - Challenge: shrinkage as a convex combination

Start from $\text{score} = \dfrac{v\,R + m\,C}{v + m}$ and split the numerator:

$$\text{score} = \frac{v}{v+m}\,R + \frac{m}{v+m}\,C = w\,R + (1-w)\,C, \qquad w = \frac{v}{v+m}.$$

For $v \ge 0$ and $m > 0$ we have $0 \le w < 1$ (and $w \le 1$ allowing $m = 0$), so the score is a genuine convex combination of $R$ and $C$ -- it always lies between them.

Limits: as $m \to 0$, $w \to 1$ and $\text{score} \to R$ (raw rate); as $m \to \infty$, $w \to 0$ and $\text{score} \to C$ (global rate). Since $\text{score} - C = w\,(R - C) = \dfrac{v\,(R-C)}{v+m}$ and $w$ decreases monotonically in $m$, every score contracts toward $C$ as $m$ grows.

Head-share direction: as $m \to \infty$ the ranking approaches ordering by $v\,(R - C)$ -- an exposure-weighted "how far above average". Because $v$ (exposure) tracks popularity, that order is dominated by the same high-exposure **head** items, so head-share should **rise** as $m$ grows. At the raw rate ($m \to 0$) the ranker instead floats lightly-exposed tail items to the top (a perfect rate on a handful of observations beats a $0.9$ on ten thousand), so head-share starts near $0$; as $m$ grows the slots fill with head items and head-share climbs toward $1.0$. Below we verify the convex-combination identity numerically for $m = 10$ on the demo arrays, then measure the whole-catalog quality ranker's head-share at a small and a large $m$ (seed $=0$ substrate) to confirm the rise.

In [ ]:
v = np.array([3.0, 50.0, 10000.0, 1.0])
R = np.array([1.0, 0.8, 0.9, 1.0])
C_demo = 0.5
m = 10.0

w = v / (v + m)                                              # convex weight on R
score_direct = bookrec.weighted_rating((v * R).astype(int), v.astype(int), m=m, global_rate=C_demo)
score_convex = w * R + (1 - w) * C_demo
identity_holds = bool(np.allclose(score_direct, score_convex))
w_in_unit_interval = bool(np.all((w >= 0.0) & (w <= 1.0)))
head_share_direction = (
    "rise - as m grows the order approaches v*(R-C), and v tracks popularity, so the ranking "
    "concentrates on the same high-exposure head items; head-share climbs from ~0 at the raw "
    "rate (which floats lightly-exposed tail items to the top) toward 1.0 as m gets large."
)

# measure head-share of the whole-catalog quality ranker at a small vs large m (seed=0 substrate)
head = bookrec.head_ids_from_counts(popularity.counts, catalog_ids, fraction=0.10)
raw_quality = bookrec.weighted_rating(item_positives, item_exposures, m=0.0, global_rate=C_global)
big_quality = bookrec.weighted_rating(item_positives, item_exposures, m=1000.0, global_rate=C_global)
raw_head_share = bookrec.head_share(
    scored_recommendations(QualityRetrievalPath(catalog_ids, raw_quality)), head
)
big_head_share = bookrec.head_share(
    scored_recommendations(QualityRetrievalPath(catalog_ids, big_quality)), head
)

print(f"score_direct = {np.round(score_direct, 6)}")
print(f"score_convex = {np.round(score_convex, 6)}")
print(f"w = {np.round(w, 4)}  identity_holds={identity_holds}  w_in_unit_interval={w_in_unit_interval}")
print(f"head-share: raw rate (m->0) = {raw_head_share:.4f}   large m = {big_head_share:.4f}")

assert identity_holds, "the weighted rating must equal the convex combination w*R + (1-w)*C"
assert w_in_unit_interval, "every convex weight must lie in [0, 1]"
assert big_head_share > raw_head_share, "head-share must rise as larger m concentrates the ranking on the head"
assert raw_head_share < 0.5 < big_head_share, "raw rate floats tail items; large m fills slots with the head"

## Exercise 8 - Challenge: find the m that makes two items tie

Set the two weighted ratings equal and cross-multiply:

$$(v_1 R_1 + m C)(v_2 + m) = (v_2 R_2 + m C)(v_1 + m).$$

Expanding, the $m^2 C$ term appears on both sides and cancels. Collecting the remaining terms:

$$v_1 v_2 R_1 + m(v_1 R_1 + C v_2) = v_1 v_2 R_2 + m(v_2 R_2 + C v_1),$$

so $v_1 v_2 (R_1 - R_2) = m\big[(v_2 R_2 - v_1 R_1) + C(v_1 - v_2)\big]$, giving the closed form

$$m^\star = \frac{v_1 v_2 (R_1 - R_2)}{(v_2 R_2 - v_1 R_1) + C\,(v_1 - v_2)}.$$

Three cases decide what this closed form means:

- **Numerator and denominator both non-zero and sharing a sign** -- one finite positive $m^\star$: the ranking crosses once, and past $m^\star$ the two items swap. (If they are non-zero with *opposite* signs, $m^\star \le 0$: no positive prior can swap an item that is already both deeper and better -- shrinkage only ever pulls both toward $C$.)
- **Denominator zero, numerator non-zero** -- no finite solution: the two scores approach a common limit only as $m \to \infty$ and never meet at any finite $m$.
- **Both numerator and denominator zero** -- the two weighted ratings are equal for *every* $m$: a degenerate identity, not a crossover. This happens when the items are indistinguishable to the formula (e.g. $v_1 = v_2$ and $R_1 = R_2$, or both rates already equal $C$), so there is no single tie-$m$ to report -- they tie at all prior strengths.

With the given numbers item 1 is thin-but-perfect ($3/3$) and item 2 is deep ($9000/10000$), so numerator and denominator share a sign and a small positive $m^\star$ makes them tie, after which item 2 overtakes.

In [ ]:
v1, R1 = 3, 1.0
v2, R2 = 10000, 0.9
C = 0.5

m_star = (v1 * v2 * (R1 - R2)) / ((v2 * R2 - v1 * R1) + C * (v1 - v2))
exists_condition = (
    "Let num = v1*v2*(R1-R2) and den = (v2*R2 - v1*R1) + C*(v1-v2). A finite positive m* exists "
    "only when num and den are both non-zero and share a sign. Non-zero with opposite signs gives "
    "m*<=0 (no positive prior swaps an already-deeper-and-better item). den==0 with num!=0 gives no "
    "finite m* (the scores meet only in the m->infinity limit). num==0 AND den==0 means the two "
    "weighted ratings are equal for every m -- a degenerate identity (the items are already "
    "indistinguishable to the formula), so there is no single tie-m."
)

pos = np.array([int(round(v1 * R1)), int(round(v2 * R2))])
exp = np.array([v1, v2])
scores_at_m_star = bookrec.weighted_rating(pos, exp, m=m_star, global_rate=C)
scores_agree = bool(np.isclose(scores_at_m_star[0], scores_at_m_star[1]))

print(f"m_star = {m_star:.6f}")
print(f"scores at m_star = {np.round(scores_at_m_star, 6)}  scores_agree={scores_agree}")

assert m_star > 0.0, "a finite positive crossover m exists for a thin-perfect vs deep item"
assert scores_agree, "the two weighted ratings must tie exactly at m_star"